# Estrutura tabular, limpeza e qualidade

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/02_estrutura_limpeza_e_qualidade.ipynb)

O Notebook 01 produziu representações legíveis e um inventário de riscos.
Agora precisamos organizar essas representações segundo unidades explícitas e
aplicar correções sem apagar os valores recebidos ou suas incertezas.

## 1. Uma variável por coluna, uma observação por linha

A estrutura tabular depende da unidade de análise. Valores atômicos e nomes de
campos estáveis facilitam seleção e junção, mas não determinam sozinhos a
ontologia correta. Tabelas separadas podem representar documentos, pessoas,
lugares e relações sem repetir tudo em uma única linha.

In [1]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_03'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('openpyxl', 'openpyxl>=3.1,<4'), ('pypdf', 'pypdf>=5,<6'), ('PIL', 'Pillow>=11,<12')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente local: nenhuma clonagem necessária.


In [2]:
import pandas as pd
bruto = pd.read_csv("dados/brutos/catalogo_messy.csv", sep=";", dtype={"codigo_municipio": "string"})
bruto.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_documento      8 non-null      object 
 1   titulo            8 non-null      object 
 2   data_documento    8 non-null      object 
 3   municipio         8 non-null      object 
 4   codigo_municipio  8 non-null      string 
 5   genero            8 non-null      object 
 6   palavras          7 non-null      float64
dtypes: float64(1), object(5), string(1)
memory usage: 580.0+ bytes


A inspeção da tabela mostra que colunas e linhas já carregam decisões sobre
entidades e observações. A comparação entre formatos largo e longo torna
visível como uma reorganização pode mudar precisamente a unidade da linha.

## 2. Largo e longo

No formato largo, período e tema aparecem nos nomes das colunas. No longo,
essas dimensões viram valores. A transformação altera a unidade da linha: de
documento para combinação documento–tema–período.

![Uma tabela larga com uma linha por documento é reorganizada em tabela longa com uma linha por combinação documento, tema e período, sem resumir os valores.](imagens/02_largo_longo.svg)

Observe que `melt` não apenas muda a aparência: ele redefine o que cada linha
representa. Essa unidade precisa constar no dicionário de dados.

In [3]:
largo = pd.read_csv("dados/brutos/indicadores_largos.csv")
longo = largo.melt(id_vars="id_documento", var_name="tema_periodo", value_name="ocorrencias")
partes = longo["tema_periodo"].str.extract(r"(?P<tema>.+)_(?P<periodo>\d{4})")
longo = pd.concat([longo[["id_documento", "ocorrencias"]], partes], axis=1)
print("Largo:", largo.shape, "| Longo:", longo.shape)
longo.head(6)

Largo: (3, 5) | Longo: (12, 4)


,id_documento,ocorrencias,tema,periodo
0,D001,3,educacao,1890
1,D002,0,educacao,1890
2,D003,2,educacao,1890
3,D001,1,trabalho,1890
4,D002,4,trabalho,1890
5,D003,0,trabalho,1890


A transformação para o formato longo reorganiza valores sem precisar apagar a
entrada. O mesmo princípio vale para correções textuais: em vez de substituir
silenciosamente, criaremos uma representação normalizada ao lado do original.

## 3. Preservar original, criar versão normalizada

Não sobrescreveremos títulos, municípios ou gêneros. A coluna original permite
auditar o mapa de equivalências e recuperar distinções apagadas por uma regra.
A remoção de acentos pode ajudar correspondência aproximada, mas não deve
substituir automaticamente a grafia de apresentação.

In [4]:
import re
import unicodedata

def chave_textual(valor):
    if pd.isna(valor):
        return pd.NA
    texto = " ".join(str(valor).strip().lower().split())
    texto = "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")
    return texto

trabalho = bruto.copy()
trabalho["titulo_chave"] = trabalho["titulo"].map(chave_textual)
trabalho["municipio_chave"] = trabalho["municipio"].map(chave_textual)
mapa_generos = {"editorial": "editorial", "notícia": "notícia", "noticia": "notícia", "carta": "carta", "manifesto": "manifesto"}
trabalho["genero_original"] = trabalho["genero"]
trabalho["genero_padronizado"] = trabalho["genero"].map(chave_textual).map(mapa_generos)
trabalho[["titulo", "titulo_chave", "genero_original", "genero_padronizado"]]

,titulo,titulo_chave,genero_original,genero_padronizado
0,Jornal Aurora,jornal aurora,Editorial,editorial
1,gazeta popular,gazeta popular,notícia,notícia
2,Correio do Vale,correio do vale,Carta,carta
3,JORNAL AURORA,jornal aurora,editorial,editorial
4,Gazeta Popular,gazeta popular,Noticia,notícia
5,Correio do Vale,correio do vale,carta,carta
6,Correio do Vale,correio do vale,Carta,carta
7,Boletim Operário,boletim operario,manifesto,manifesto


As chaves normalizadas facilitam comparações, mas cada tipo de campo exige uma
regra própria. Datas incompletas, identificadores numéricos e valores ausentes
não podem ser tratados como simples variações ortográficas.

## 4. Datas, códigos e ausências

Código de município é identificador textual, não quantidade. Datas parciais
não devem receber dia e mês inventados. Um parser explícito separa datas
completas, anos isolados e valores desconhecidos; guardar o original, a
precisão e a razão evita transformar incerteza em falsa exatidão.

![O valor de data recebido é preservado, passa por uma regra explícita e produz representação derivada, precisão e razão; log e testes permitem retornar à fonte.](imagens/02_transformacao_rastreavel.svg)

No exemplo, uma data completa pode ocupar `data_normalizada`; um ano isolado
ocupa `ano_documento`, mas não recebe dia e mês arbitrários. A coluna
`precisao_data` distingue `dia`, `ano`, `desconhecida` e, se ocorrer, `inválida`.

In [5]:
trabalho["data_original"] = trabalho["data_documento"]

def decompor_data(valor):
    texto = str(valor).strip()
    formatos = [
        (r"\d{4}-\d{2}-\d{2}", "%Y-%m-%d"),
        (r"\d{2}/\d{2}/\d{4}", "%d/%m/%Y"),
    ]
    for padrao, formato in formatos:
        if re.fullmatch(padrao, texto):
            data = pd.to_datetime(texto, format=formato, errors="coerce")
            if pd.notna(data):
                return data, data.year, "dia", pd.NA
            return pd.NaT, pd.NA, "inválida", "data impossível no calendário"
    if re.fullmatch(r"\d{4}", texto):
        return pd.NaT, int(texto), "ano", "dia e mês não informados"
    if texto.casefold() == "data desconhecida":
        return pd.NaT, pd.NA, "desconhecida", "data não informada"
    return pd.NaT, pd.NA, "inválida", "formato não reconhecido"

datas = trabalho["data_original"].apply(decompor_data).apply(pd.Series)
datas.columns = ["data_normalizada", "ano_documento", "precisao_data", "razao_data_ausente"]
datas["ano_documento"] = datas["ano_documento"].astype("Int64")
trabalho[datas.columns] = datas
trabalho["palavras"] = pd.to_numeric(trabalho["palavras"], errors="coerce")
trabalho["razao_palavras_ausente"] = trabalho["palavras"].isna().map({True: "não contado", False: pd.NA})
trabalho[["data_original", "data_normalizada", "ano_documento", "precisao_data", "razao_data_ausente", "palavras"]]

,data_original,data_normalizada,ano_documento,precisao_data,razao_data_ausente,palavras
0,1890-01-05,1890-01-05,1890,dia,<NA>,860.0
1,06/02/1891,1891-02-06,1891,dia,<NA>,610.0
2,1892,NaT,1892,ano,dia e mês não informados,NaN
3,1893-03-12,1893-03-12,1893,dia,<NA>,940.0
4,data desconhecida,NaT,<NA>,desconhecida,data não informada,720.0
5,1900-08-20,1900-08-20,1900,dia,<NA>,420.0
6,20/08/1900,1900-08-20,1900,dia,<NA>,420.0
7,1901-04-09,1901-04-09,1901,dia,<NA>,500.0


Depois de normalizar campos e representar ausências, podemos comparar registros
com maior consistência. Ainda assim, semelhança não prova identidade: ela apenas
produz candidatos a duplicata que precisam de revisão documental.

## 5. Duplicatas são uma hipótese

IDs repetidos detectam um tipo de duplicata. Registros de um mesmo documento
com IDs diferentes exigem combinação de campos e revisão. Remover pelo título
isolado poderia apagar edições legítimas.

In [6]:
trabalho["possivel_duplicata"] = trabalho.duplicated(
    subset=["titulo_chave", "data_normalizada", "municipio_chave", "palavras"],
    keep=False,
)
trabalho.loc[trabalho["possivel_duplicata"], ["id_documento", "titulo", "data_original", "palavras"]]

,id_documento,titulo,data_original,palavras
5,D006,Correio do Vale,1900-08-20,420.0
6,D006-copia,Correio do Vale,20/08/1900,420.0


As regras anteriores modificaram representações, registraram incertezas e
sinalizaram casos para revisão. Antes de usar essa tabela em junções, precisamos
resumir o que mudou e exportar uma camada intermediária reproduzível.

## 6. Relatório e exportação intermediária

Antes/depois deve quantificar transformações, falhas e casos para revisão. A
saída intermediária não substitui os dados brutos.

In [7]:
relatorio = {
    "linhas": len(trabalho),
    "datas_sem_precisao_de_dia": int(trabalho["data_normalizada"].isna().sum()),
    "datas_parciais_ano": int(trabalho["precisao_data"].eq("ano").sum()),
    "datas_desconhecidas": int(trabalho["precisao_data"].eq("desconhecida").sum()),
    "datas_invalidas": int(trabalho["precisao_data"].eq("inválida").sum()),
    "palavras_ausentes": int(trabalho["palavras"].isna().sum()),
    "generos_sem_mapeamento": int(trabalho["genero_padronizado"].isna().sum()),
    "registros_em_grupos_de_possiveis_duplicatas": int(trabalho["possivel_duplicata"].sum()),
}
trabalho.to_csv("dados/intermediarios/catalogo_normalizado.csv", index=False)
pd.Series(relatorio, name="quantidade")

linhas                                         8
datas_sem_precisao_de_dia                      2
datas_parciais_ano                             1
datas_desconhecidas                            1
datas_invalidas                                0
palavras_ausentes                              1
generos_sem_mapeamento                         0
registros_em_grupos_de_possiveis_duplicatas    2
Name: quantidade, dtype: int64

## Aplicação ao projeto Goodreads

As células seguintes aplicam ao arquivo do projeto as mesmas ideias do
notebook: preservar o original, criar versões normalizadas com regra explícita,
sinalizar em vez de corrigir e tratar duplicatas como hipótese. As funções
estão em `projeto_goodreads/preparar_base.py`.

In [8]:
# Localiza a pasta do projeto Goodreads (localmente ou no Colab, se a pasta for enviada para /content)
import sys
from pathlib import Path

CANDIDATOS = [Path("../projeto_goodreads"), Path("projeto_goodreads"), Path("/content/projeto_goodreads")]
PROJETO = next((c.resolve() for c in CANDIDATOS if (c / "preparar_base.py").exists()), None)
if PROJETO is None:
    raise FileNotFoundError(
        "Pasta projeto_goodreads não encontrada. Coloque-a ao lado das pastas unidade_0X "
        "ou, no Colab, envie-a para /content."
    )
if str(PROJETO) not in sys.path:
    sys.path.insert(0, str(PROJETO))
print("Projeto localizado:", (PROJETO / "preparar_base.py").exists())

from preparar_base import ler_bruto, normalizar, chave_textual

gr_bruto = ler_bruto()
gr = normalizar(gr_bruto)

# 1. Uma regra de chave textual mal desenhada cria falsas duplicatas
def chave_ascii(valor):
    texto = re.sub(r"\s*\(.*?\)\s*", " ", str(valor))
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(re.findall(r"[a-z0-9]+", texto))

gr["titulo_chave_ascii"] = gr["titulo"].map(chave_ascii)
print("Livros em grupos de títulos repetidos, regra ASCII:", int(gr.duplicated("titulo_chave_ascii", keep=False).sum()))
print("Livros em grupos de títulos repetidos, regra Unicode:", int(gr.duplicated("titulo_chave", keep=False).sum()))
print("Possíveis duplicatas (mesma chave e mesmo autor):", int(gr["possivel_duplicata"].sum()))
gr.loc[gr["titulo_repetido_outro_autor"], ["titulo", "autor", "ano_publicacao"]].sort_values("titulo").head(8)

Projeto localizado: True


Livros em grupos de títulos repetidos, regra ASCII: 139
Livros em grupos de títulos repetidos, regra Unicode: 18
Possíveis duplicatas (mesma chave e mesmo autor): 0


,titulo,autor,ano_publicacao
4995,Angels Flight,Michael Connelly,1999
2258,Angels' Flight,Nalini Singh,2012
9125,Blaze,Wolf Specter,2016
3600,Blaze (or Love in the Time of Supervillains),Laurie Boyle Crompton,2013
8006,Fire & Ice,Alice Brown,2014
10062,Fire Ice,Clive Cussler,2002
3232,Haunted,Kay Hooper,2014
789,Haunted (State v. Sefore #1),Charity Tinnin,2014


**Leitura.** A regra ASCII coloca 139 livros em grupos de títulos
repetidos. Quase todos são títulos em árabe ou persa, cujas letras a regra
apaga, deixando chaves vazias ou reduzidas a fragmentos iguais. A regra Unicode
reduz o número para 18 livros, que são de fato títulos iguais ou muito
parecidos. Mesmo com a
regra correta, títulos iguais de autores diferentes são obras diferentes.
Nenhum livro foi removido.

In [9]:
# 2. Padronização do local de nascimento: original preservado, regra registrada
print(gr["regra_pais"].value_counts())
gr.loc[gr["regra_pais"].ne("mantido"), ["pais_original", "pais_padronizado", "regra_pais"]].drop_duplicates("pais_original").head(12)

regra_pais
mantido                                                                    10159
cidade ou estado dos EUA agregado ao país                                    147
nação constituinte ou cidade agregada ao Reino Unido                          91
entidade histórica ou nome incompleto mantido sem correspondência atual       36
ambíguo: pode corresponder a mais de um país                                   8
equivalência direta documentada                                                7
inválido: informa residência, não nascimento                                   2
inválido: local implausível para nascimento, sem confirmação                   1
inválido: identidade da autoria declarada como oculta                          1
inválido: valor ficcional ou humorístico                                       1
inválido: URL no lugar do local                                                1
Name: count, dtype: int64


,pais_original,pais_padronizado,regra_pais
51,Colorado,United States,cidade ou estado dos EUA agregado ao país
77,Austria-Hungary,Austria-Hungary,entidade histórica ou nome incompleto mantido ...
101,England,United Kingdom,nação constituinte ou cidade agregada ao Reino...
131,Alabama,United States,cidade ou estado dos EUA agregado ao país
166,Seoul,South Korea,equivalência direta documentada
173,Southern California,United States,cidade ou estado dos EUA agregado ao país
189,New York,United States,cidade ou estado dos EUA agregado ao país
211,New Jersey,United States,cidade ou estado dos EUA agregado ao país
224,KY,United States,cidade ou estado dos EUA agregado ao país
333,TX,United States,cidade ou estado dos EUA agregado ao país


In [10]:
# 3. Sinalizações (nada é alterado)
pd.Series({
    "anos_antes_de_1900": int(gr["ano_antes_de_1900"].sum()),
    "anos_negativos_aC": int((gr["ano_publicacao"] < 0).sum()),
    "paginas_suspeitas": int(gr["paginas_suspeitas"].sum()),
    "livro_com_mais_avaliacoes_que_o_autor": int(gr["contagem_livro_maior_que_autor"].sum()),
    "titulos_sem_letra_latina": int(gr["titulo_sem_letra_latina"].sum()),
    "titulos_com_marcador_de_serie": int(gr["titulo_tem_marcador_serie"].sum()),
})

anos_antes_de_1900                       327
anos_negativos_aC                         19
paginas_suspeitas                         29
livro_com_mais_avaliacoes_que_o_autor     38
titulos_sem_letra_latina                 129
titulos_com_marcador_de_serie             40
dtype: int64

### Auditoria do rótulo de gênero da autoria

O rótulo `genero_autor` chegou sem regra documentada. Para avaliá-lo,
comparo-o com a classificação usual do primeiro prenome em um dicionário de
nomes (pacote `gender-guesser`, que usa o dicionário de Jörg Michael). A
comparação é feita por autor, não por livro. Iniciais (J. R., C. S.) não são
classificadas: nesses casos o resultado é `sem_prenome`.

In [11]:
from preparar_base import inferir_genero_por_nome

autores_gr = gr.drop_duplicates("id_autor")[["id_autor", "autor", "genero_autor", "n_avaliacoes_autor"]]
auditoria_nome = inferir_genero_por_nome(autores_gr[["id_autor", "autor"]])
autores_gr = autores_gr.merge(auditoria_nome, on="id_autor", validate="one_to_one")

tabela_auditoria = pd.crosstab(autores_gr["genero_por_nome"], autores_gr["genero_autor"], margins=True)
display(tabela_auditoria)
proporcao = pd.Series({
    "prenomes femininos rotulados como masculino": ((autores_gr["genero_por_nome"] == "female") & (autores_gr["genero_autor"] == "masculino")).sum() / (autores_gr["genero_por_nome"] == "female").sum(),
    "prenomes masculinos rotulados como feminino": ((autores_gr["genero_por_nome"] == "male") & (autores_gr["genero_autor"] == "feminino")).sum() / (autores_gr["genero_por_nome"] == "male").sum(),
}).round(3)
proporcao

genero_autor,feminino,masculino,All
genero_por_nome,,,
andy,35,42,77
female,2344,886,3230
male,122,2588,2710
mostly_female,209,108,317
mostly_male,81,140,221
sem_prenome,233,250,483
unknown,289,453,742
All,3313,4467,7780


prenomes femininos rotulados como masculino    0.274
prenomes masculinos rotulados como feminino    0.045
dtype: float64

In [12]:
# Casos mais avaliados em cada sentido da divergência (para inspeção manual)
duvidosos = autores_gr[(autores_gr["genero_autor"] == "masculino") & autores_gr["genero_por_nome"].isin(["female", "mostly_female"])]
inversos = autores_gr[(autores_gr["genero_autor"] == "feminino") & autores_gr["genero_por_nome"].isin(["male", "mostly_male"])]
print("Rótulo 'masculino' com prenome feminino:", len(duvidosos), "autores")
display(duvidosos.nlargest(12, "n_avaliacoes_autor")[["autor", "genero_autor", "prenome_usado", "genero_por_nome"]])
print("Rótulo 'feminino' com prenome masculino:", len(inversos), "autores")
inversos.nlargest(12, "n_avaliacoes_autor")[["autor", "genero_autor", "prenome_usado", "genero_por_nome"]]

Rótulo 'masculino' com prenome feminino: 994 autores


,autor,genero_autor,prenome_usado,genero_por_nome
99,Janet Evanovich,masculino,Janet,female
558,Lois Lowry,masculino,Lois,female
546,Diana Gabaldon,masculino,Diana,female
560,Alice Sebold,masculino,Alice,female
86,Kiera Cass,masculino,Kiera,female
1108,Paula Hawkins,masculino,Paula,female
4329,Audrey Niffenegger,masculino,Audrey,female
89,Marissa Meyer,masculino,Marissa,female
178,Sarah Dessen,masculino,Sarah,female
306,Kristin Hannah,masculino,Kristin,female


Rótulo 'feminino' com prenome masculino: 203 autores


,autor,genero_autor,prenome_usado,genero_por_nome
579,Harper Lee,feminino,Harper,male
1002,Robin Hobb,feminino,Robin,mostly_male
1104,Robert Galbraith,feminino,Robert,male
321,Toni Morrison,feminino,Toni,male
1617,Marian Keyes,feminino,Marian,male
3351,Jean M. Auel,feminino,Jean,male
2535,Jude Deveraux,feminino,Jude,mostly_male
299,Robin McKinley,feminino,Robin,mostly_male
1380,George Eliot,feminino,George,male
518,Jay Crownover,feminino,Jay,male


**Leitura da auditoria.** As duas divergências não têm o mesmo
significado. No primeiro sentido (rótulo “masculino”, prenome feminino), os
autores mais avaliados são escritoras publicamente conhecidas como mulheres
(por exemplo, Janet Evanovich, Diana Gabaldon, Alice Sebold, Paula Hawkins e
Sarah Dessen): o **rótulo** parece errado. No sentido inverso, a maior parte dos
casos mais avaliados parece ser de mulheres cujo prenome o dicionário classifica
como masculino ou que usam pseudônimo masculino (Harper Lee, Toni Morrison,
George Eliot, Robert Galbraith): ali é a **inferência por nome** que erra. Essa
leitura se baseia em conhecimento público sobre autores famosos e deve ser
conferida pelo pesquisador nas biografias. Ela basta para concluir que o rótulo
original não pode ser usado sem verificação.

A especificação principal (E2) exclui apenas os livros do primeiro sentido,
em que a evidência aponta erro do rótulo. As alternativas E1 (rótulo original)
e E3 (reclassificação desses casos como feminino) permitirão verificar se as
conclusões dependem da decisão.

O relatório quantitativo informa quantos casos foram afetados, mas não explica
por que cada decisão foi tomada. O log acrescenta regra, justificativa, teste,
reversibilidade e responsabilidade a esse resumo.

## U03-A03 — Atividade — log de transformação

As regras foram implementadas em `projeto_goodreads/preparar_base.py` e
demonstradas nas células seguintes. Responsável por todas: o estudante.

**Log:**

| Campo | Problema | Regra | Justificativa | Valores afetados | Teste | Reversível? | Distinção que a regra pode apagar |
|---|---|---|---|---|---|---|---|
| `titulo` | comparar títulos com grafias diferentes | `titulo_chave`: retira marcador de série entre parênteses, acentos, caixa e pontuação, preservando letras de qualquer alfabeto (`\w`) | detectar possíveis duplicatas | 10.454 chaves criadas; o original fica intacto | comparar a regra ASCII (cria falsas duplicatas entre títulos árabes e persas) com a regra Unicode | sim (coluna original preservada) | “Angels' Flight” e “Angels Flight” ganham a mesma chave, embora sejam livros diferentes de autores diferentes |
| `titulo` e `id_autor` | possíveis duplicatas | duplicata = mesma `titulo_chave` **e** mesmo `id_autor` | título igual de autores diferentes é outra obra | 0 duplicatas; 18 livros com título repetido de outro autor, mantidos e sinalizados | contagem de grupos | sim | adaptações com o mesmo título (*The Lion, the Witch and the Wardrobe*, de C. S. Lewis e de Hiawyn Oram) |
| `pais_nascimento` | mistura de países, estados, cidades, entidades históricas e valores inválidos | `pais_padronizado` por tabela explícita; `regra_pais` registra a regra | permitir agrupar por país | 147 livros com cidade ou estado dos EUA; 91 com nação ou cidade do Reino Unido; 7 equivalências diretas; 36 entidades históricas mantidas; 8 ambíguos e 6 inválidos tornados ausentes | contagem por regra; todos os valores recebem uma regra | sim | Inglaterra e Escócia somem dentro do Reino Unido; URSS e Império Russo não são convertidos em Rússia para não criar anacronismo |
| `ano_publicacao` | anos negativos e muito antigos | nenhum valor alterado; `ano_antes_de_1900` e `periodo_publicacao` criados | o ano negativo indica obra antes de Cristo, não erro | 327 livros antes de 1900 | contagem por período | sim | não se sabe se o ano é da primeira edição ou da edição catalogada |
| `paginas` | valores de 1 a 6.576 | apenas sinalizar (≤ 10 ou > 2.000) | depende da edição; não entra na medida de reconhecimento | 29 livros | contagem | sim | coletâneas legítimas e erros recebem o mesmo sinal |
| `n_avaliacoes` × `n_avaliacoes_autor` | livro com mais avaliações que o total do autor | sinalizar `contagem_livro_maior_que_autor` | o total do autor deveria incluir o livro | 38 livros | comparação direta | sim | homônimos e datas de coleta diferentes não são distinguidos |
| `genero_autor` | rótulo binário de origem desconhecida | manter original; criar `genero_por_nome` (dicionário de nomes), `rotulo_duvidoso` e três especificações (E1 original, E2 sem duvidosos, E3 reclassificada) | auditar a validade do rótulo antes de comparar grupos | 994 autores e 1.274 livros com rótulo “masculino” e prenome classificado como feminino | tabela cruzada rótulo × nome; inspeção dos casos mais avaliados | sim, nada é sobrescrito | a inferência por nome não diz o gênero de ninguém; erra pseudônimos (George Eliot, Robert Galbraith) e nomes não ocidentais |
| `n_resenhas`, `n_avaliacoes` | distribuições muito assimétricas | criar `taxa_resenhas`, `log10_avaliacoes` e `log10_resenhas_mais_1` | comparar engajamento relativo e visualizar escalas | 10.454 | resenhas nunca excedem avaliações (teste passou) | sim | a razão esconde o tamanho absoluto; o log comprime diferenças grandes |

**Principal achado da etapa:** a auditoria do rótulo `genero_autor` mostrou um
erro **sistemático e assimétrico**. Entre os 3.230 autores cujo prenome o
dicionário classifica como feminino, 886 (27,4%) estão rotulados como
“masculino”. No sentido inverso, só 122 dos 2.710 prenomes masculinos (4,5%)
estão rotulados como “feminino”, e vários desses rótulos parecem corretos
(pseudônimos masculinos de mulheres, como George Eliot e Robert Galbraith, ou
autoras como Harper Lee, cujo prenome o dicionário classifica como masculino).
Entre os casos mais avaliados rotulados como “masculino” estão escritoras
conhecidas publicamente como mulheres, como Janet Evanovich, Diana Gabaldon,
Alice Sebold e Paula Hawkins. A consequência é que o grupo “masculino” do
arquivo contém um número relevante de livros de mulheres. Decisão: manter o
rótulo original e comparar três especificações nas Unidades 4 e 5. A
especificação principal (E2) exclui os 1.274 livros com rótulo duvidoso; E1
(original) e E3 (reclassificação dos duvidosos como feminino) servem de
sensibilidade.

Santamaría e Mihaljević (2018) avaliaram serviços de inferência de gênero por
nome e relatam, para o pacote `gender-guesser`, poucos erros de classificação
mas muitos nomes não classificados, além de desempenho pior em nomes asiáticos.
Por isso a ferramenta é usada aqui só para **sinalizar** casos, nunca para
atribuir gênero a uma pessoa.

O log da atividade liga cada valor derivado ao original e à regra aplicada. A
síntese recupera essa relação antes de a tabela intermediária participar de
junções com outras entidades.

## Síntese

Limpeza responsável acrescenta rastreabilidade. Ela não transforma incerteza
substantiva em certeza técnica nem autoriza exclusão silenciosa. No Notebook
03, a tabela intermediária e seu log serão usados para declarar chaves,
cardinalidades e vínculos entre documentos, textos, temas e indicadores.